<a href="https://colab.research.google.com/github/yoonsoo2002/-/blob/main/delay_prediction_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1️⃣ Importing Library

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline
import joblib

# 2️⃣ Loading Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/olist_data'
df_merged = pd.read_csv(f'{BASE_PATH}/data.csv')

Mounted at /content/drive


# 3️⃣ Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.pipeline import Pipeline
import joblib

features = ['price', 'freight_value', 'total_order_value', 'estimated_duration', 'product_volume_cm3']
cat_cols = ['product_category_name_english', 'customer_state', 'seller_state', 'purchase_day_of_week']

# Convert date columns to datetime objects
df_merged['order_estimated_delivery_date'] = pd.to_datetime(df_merged['order_estimated_delivery_date'])
df_merged['order_purchase_timestamp'] = pd.to_datetime(df_merged['order_purchase_timestamp'])

df_merged['estimated_duration'] = (
    df_merged['order_estimated_delivery_date'] - df_merged['order_purchase_timestamp']
).dt.days

x = df_merged[features + cat_cols].copy()
y = df_merged['is_late']

# 결측치 처리
x[cat_cols] = x[cat_cols].fillna('Unknown')
x[features] = x[features].fillna(0)

try:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    cat_transform = OneHotEncoder(handle_unknown='ignore', sparse=False)

preproc = ColumnTransformer([
    ('num', 'passthrough', features),
    ('cat', cat_transform, cat_cols)
])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=24,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

pipe = Pipeline([
    ('preprocessor', preproc),
    ('model', clf)
])

pipe.fit(x_train, y_train)

joblib.dump(pipe, "delay_prediction_model.pkl")
print("\n📂📂📂 Model is saved.")
print("-" * 120)

pred = pipe.predict(x_test)
pred_proba = pipe.predict_proba(x_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, pred))
print(f"ROC-AUC : {roc_auc_score(y_test, pred_proba):.4f}")
print(f"PR-AUC  : {average_precision_score(y_test, pred_proba):.4f}")
print(f"Train Score : {pipe.score(x_train, y_train):.4f}")
print(f"Test Score  : {pipe.score(x_test, y_test):.4f}")
print("_" * 120)


📂📂📂 Model is saved.
------------------------------------------------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.92      0.93     22613
           1       0.22      0.28      0.25      1911

    accuracy                           0.87     24524
   macro avg       0.58      0.60      0.59     24524
weighted avg       0.88      0.87      0.88     24524

ROC-AUC : 0.7093
PR-AUC  : 0.1880
Train Score : 0.9318
Test Score  : 0.8685
________________________________________________________________________________________________________________________
